# transistor：検証画像の回転に対する異常検知性能の比較

保存済みHybrid CNN＋FCの**通常学習／回転拡張（両方6,700更新）**を評価します。学習は一切行いません。

| 評価条件 | 内容 | 元画像数／評価サンプル数 |
| --- | --- | --- |
| 回転なし | 元の100枚。既存結果と照合 | 100／100 |
| 角度別 | 同じ100枚を各角度に回転。10角度を個別評価 | 各100／100 |
| 向きが混在 | 元画像ごとに1角度を固定して割り当てる | 100／100 |
| 全角度（補足） | 全画像×全角度をまとめて評価 | 100／延べ1,000 |

0〜324度を36度刻みで評価します。向きが混在する条件は、正常を各角度6枚、各異常種類を各角度1枚とし、seed42で画像と角度の対応を固定します。角度別の結果の平均と、全角度をまとめたAUROCは別の量です。

**注意：** 合成回転は実際の撮影条件変化そのものではありません。背景補完・画面端の切り取り・補間の影響も含みます。画像全体の回転を撮影方向の変化と仮定し、元の正常／異常ラベルを維持します。特に「配置ずれ」は、対象と背景の相対配置の異常と、撮影方向の変化を区別して解釈してください。欠陥が切り取られる可能性もあります。

元のテスト結果を確認した後の探索的な追試です。同一画像の派生サンプルを独立な1,000枚と数えず、実環境での優位性や回転拡張の改善を事前には仮定しません。

## 実行方法
設定を確認して上から実行します。学習スイッチ・学習処理はありません。毎回新規の評価フォルダへ保存します。アシスタントはコード作成・静的確認のみ行い、実行はユーザーが行います。


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from io import BytesIO
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import importlib.metadata
import json
import math
import platform
import random
import uuid
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torchvision import transforms
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display
from skimage.metrics import structural_similarity as ssim

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'memory/memory.md').is_file() and (p/'data/transistor').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('このリポジトリ内を作業ディレクトリにしてください。')
NORMAL_RUN = ROOT/'saved_models/transistor_update_matched_20261106/20261106_021112_b266ecab'
ROTATION_RUN = ROOT/'saved_models/transistor_rotation_control_20261106_reflect/20261106_002435_9b4da165'
BASELINE_DIR = NORMAL_RUN/'evaluations/20261106_025126_58aad9c8'
OUTPUT_BASE = ROOT/'saved_models/transistor_rotated_test_20261106'
MODEL_PATHS = {'A_original':NORMAL_RUN/'model_final.pth',
               'B_rotate10':ROTATION_RUN/'seed_42/B_rotate10/model_final.pth'}
EXPECTED_HASHES = {'A_original':'380bb28d611612ecda31f97dee299c2310e1b39f6c52b2fbd92144e2ab3e4667',
                   'B_rotate10':'20b0568e0cbae99290cb474b7e327a95983368fb37ece7edbcd0cd1c91f42de5'}
CONDITIONS = tuple(MODEL_PATHS)
LABELS = {'A_original':'A：通常学習','B_rotate10':'B：回転拡張'}
COLORS = {'A_original':'#526b8e','B_rotate10':'darkorange'}
SCENARIO_LABELS = {'unrotated':'回転なし','mixed':'向きが混在','pooled':'全角度（補足）'}
CATEGORY_LABELS = {'good':'正常','bent_lead':'端子の曲がり','cut_lead':'端子の切断',
                   'damaged_case':'本体の損傷','misplaced':'配置ずれ'}
ANGLES = tuple(range(0,360,36))
ASSIGNMENT_SEED = 42
DEVICE = torch.device('cpu')  # 既存の評価と同じ環境
EVAL_BATCH_SIZE = 32
TOP_K = 130
# 可視化だけを絞る設定です。AUROCは常に全100画像×10角度から計算します。
VISUAL_PATHS = None  # 例：['good/000.png','cut_lead/000.png']。Noneなら種類ごとの先頭画像
MAX_IMAGES_PER_CATEGORY = 1  # VISUAL_PATHS=Noneのとき。Noneなら全画像
VISUAL_ANGLES = (0,36,72)  # Noneなら全10方向。必要な角度に変更してください。

random.seed(ASSIGNMENT_SEED)
np.random.seed(ASSIGNMENT_SEED)
torch.manual_seed(ASSIGNMENT_SEED)
torch.set_num_threads(min(4,max(1,torch.get_num_threads())))
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
for font in ['Yu Gothic','Meiryo','MS Gothic','Noto Sans CJK JP','IPAexGothic']:
    if font in available_fonts:
        plt.rcParams['font.family'] = font
        break
else:
    import japanize_matplotlib
plt.rcParams.update({'font.size':11,'axes.unicode_minus':False,'figure.dpi':100})
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))
def write_json(path,value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+chr(10),encoding='utf-8')
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


## 1. 保存済みモデル・設定・テスト画像を照合


In [ ]:
normal_config = read_json(NORMAL_RUN/'config.json')
rotation_config = read_json(ROTATION_RUN/'config.json')
normal_summary = read_json(NORMAL_RUN/'summary.json')
rotation_summary = read_json(MODEL_PATHS['B_rotate10'].parent/'summary.json')
assert normal_config['status'] == rotation_config['status'] == 'completed'
assert normal_summary['updates'] == rotation_summary['updates'] == 6700
assert normal_config['seed'] == normal_summary['seed'] == rotation_summary['seed'] == 42
assert normal_summary['initial_state_sha256'] == rotation_summary['initial_state_sha256']
assert normal_config['source_model_sha256'] == EXPECTED_HASHES['B_rotate10']
assert normal_config['model'] == 'Hybrid CNN + FC'
assert rotation_config['angles'] == list(ANGLES)
assert rotation_config['rotation_background'] == 'reflect'
assert rotation_config['rotation_interpolation'] == 'NEAREST'
assert rotation_config['rotation_expand'] is False
assert rotation_config['rotation_stage'] == 'full_resolution_before_resize'
assert rotation_config['source_size'] == [1024,1024]
assert normal_config['latent_dim'] == rotation_config['latent_dim'] == 64
assert normal_summary['model_sha256'] == EXPECTED_HASHES['A_original']
assert all(sha256(path) == EXPECTED_HASHES[name] for name,path in MODEL_PATHS.items())
source_files = [*MODEL_PATHS.values(), NORMAL_RUN/'config.json', ROTATION_RUN/'config.json',
                NORMAL_RUN/'summary.json', MODEL_PATHS['B_rotate10'].parent/'summary.json',
                ROTATION_RUN/'test_manifest.json', NORMAL_RUN/'test_manifest.json',
                BASELINE_DIR/'comparison.csv', BASELINE_DIR/'image_scores.csv']
source_hashes = {p.relative_to(ROOT).as_posix():sha256(p) for p in source_files}
baseline_comparison = pd.read_csv(BASELINE_DIR/'comparison.csv').set_index('condition')
baseline_scores = pd.read_csv(BASELINE_DIR/'image_scores.csv').set_index('relative_path')
assert np.isclose(baseline_comparison.loc['A_original','auroc'],0.82625,atol=1e-12,rtol=0)
assert np.isclose(baseline_comparison.loc['B_rotate10','auroc'],0.8066666666666666,atol=1e-12,rtol=0)


In [ ]:
# 全条件で画像全体を使う。中央820の切り出しは行わない。
final_transform=transforms.Compose([
    transforms.Resize((128,128),interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
])
to_tensor=transforms.ToTensor()
def rotation_pad_width(size):
    return math.ceil((math.hypot(*size)-min(size))/2)+4

def rotate_with_reflect(image,angle,verify_support=False):
    if angle==0: return image.copy()
    width,height=image.size
    pad=rotation_pad_width(image.size)
    pixels=np.asarray(image)
    extended=Image.fromarray(np.pad(pixels,((pad,pad),(pad,pad),(0,0)),mode='reflect'))
    box=(pad,pad,pad+width,pad+height)
    options=dict(interpolation=transforms.InterpolationMode.NEAREST,expand=False)
    out=TF.rotate(extended,angle=angle,fill=255,**options).crop(box)
    mask=TF.rotate(Image.new('L',image.size,255),angle=angle,fill=0,**options)
    original_part=TF.rotate(image,angle=angle,fill=255,**options)
    valid=np.asarray(mask)>0
    result=np.asarray(out).copy()
    result[valid]=np.asarray(original_part)[valid]
    if verify_support:
        support=TF.rotate(Image.new('L',extended.size,255),angle=angle,fill=0,**options).crop(box)
        assert np.all(np.asarray(support)==255),'パディングが不足しています。'
        assert np.array_equal(result[valid],np.asarray(original_part)[valid])
    return Image.fromarray(result)



In [ ]:
TEST_DIR = ROOT/'data/transistor/test'
expected_manifest = read_json(ROTATION_RUN/'test_manifest.json')
assert expected_manifest == read_json(NORMAL_RUN/'test_manifest.json')
paths = sorted(TEST_DIR.glob('*/*.png'),key=lambda p:p.relative_to(TEST_DIR).as_posix())
assert [p.relative_to(ROOT).as_posix() for p in paths] == [r['repo_path'] for r in expected_manifest]
base_images,base_tensors,image_records = [],[],[]
for path,record in zip(paths,expected_manifest):
    raw = path.read_bytes()
    assert hashlib.sha256(raw).hexdigest() == record['file_sha256']
    with Image.open(BytesIO(raw)) as im:
        assert im.size == (1024,1024)
        image = im.convert('RGB').copy()
    tensor = to_tensor(final_transform(image))
    assert hashlib.sha256(tensor.numpy().tobytes()).hexdigest() == record['preprocessed_sha256']
    base_images.append(image);base_tensors.append(tensor);image_records.append(record)
base_tensor = torch.stack(base_tensors)
image_table = pd.DataFrame(image_records)
counts = image_table.category.value_counts().to_dict()
assert counts == {'good':60,'bent_lead':10,'cut_lead':10,'damaged_case':10,'misplaced':10}
manifest_hash = hashlib.sha256(json.dumps(image_records,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()
assert manifest_hash == normal_config['test_manifest_sha256'] == rotation_config['test_manifest_sha256']
assert np.array_equal(baseline_scores.loc[image_table.relative_path,'true_label'],image_table.true_label)
display(pd.DataFrame([{'種類':CATEGORY_LABELS[k],'枚数':counts[k]} for k in CATEGORY_LABELS]))


## 2. 向きが混在する条件の角度を固定する

正常60枚には10角度を6回ずつ、各異常10枚には10角度を1回ずつ割り当てます。各種類内で乱数seed42により対応をシャッフルします。スコアやモデルの結果を見て角度を選びません。

これは「角度の出現割合が均等」という1つの想定です。実際の検査時の向きの頻度とは限りません。両モデルには同じ割り当てを使い、0度も含みます。


In [ ]:
rng = np.random.default_rng(ASSIGNMENT_SEED)
assigned_angles = np.empty(len(image_table),dtype=int)
for category in CATEGORY_LABELS:
    indices = image_table.index[image_table.category == category].to_numpy()
    assert len(indices) % len(ANGLES) == 0
    assigned_angles[indices] = rng.permutation(np.tile(np.array(ANGLES),len(indices)//len(ANGLES)))
assignment = image_table[['relative_path','category','category_ja','true_label']].copy()
assignment['assigned_angle'] = assigned_angles
display(pd.crosstab(assignment.category_ja,assignment.assigned_angle))

if VISUAL_PATHS is not None:
    assert len(VISUAL_PATHS) == len(set(VISUAL_PATHS))
    assert set(VISUAL_PATHS).issubset(set(image_table.relative_path)), '存在しない表示画像名があります。'
    visual_indices = image_table.index[image_table.relative_path.isin(VISUAL_PATHS)].tolist()
else:
    assert MAX_IMAGES_PER_CATEGORY is None or (isinstance(MAX_IMAGES_PER_CATEGORY,int)
           and not isinstance(MAX_IMAGES_PER_CATEGORY,bool) and MAX_IMAGES_PER_CATEGORY > 0)
    visual_indices = []
    for category in CATEGORY_LABELS:
        indices = image_table.index[image_table.category == category].tolist()
        visual_indices.extend(indices if MAX_IMAGES_PER_CATEGORY is None else indices[:MAX_IMAGES_PER_CATEGORY])
visual_angles = ANGLES if VISUAL_ANGLES is None else tuple(VISUAL_ANGLES)
assert len(set(visual_angles)) == len(visual_angles) and set(visual_angles).issubset(ANGLES)
print(f'可視化：{len(visual_indices)}画像 × {len(visual_angles)}角度。評価自体は全100画像×10角度です。')


## 3. 同じモデル構造・平均μでの推論・SSIM Top-130


In [ ]:
class VAE_Hybrid_CNN_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_Hybrid_CNN_FC, self).__init__()

        # -------------------------------------------------------------
        # 【エンコーダ前半】 CNN_Deep5（空間圧縮: 128x128 -> 4x4）
        # -------------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【エンコーダ後半】 FC_Only（段階的圧縮: 4096次元 -> 128次元）
        # -------------------------------------------------------------
        self.encoder_fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )

        # 潜在空間へのマッピング（128次元 -> 64次元）
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # -------------------------------------------------------------
        # 【デコーダ前半】 FC_Only（段階的復元: 64次元 -> 4096次元）
        # -------------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            # ※CNNへ繋ぐため、最後はSigmoidではなくReLUを使用します
            nn.Linear(1024, 256 * 4 * 4), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【デコーダ後半】 CNN_Deep5（空間復元: 4x4 -> 128x128）
        # -------------------------------------------------------------
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        # 1. CNNによる特徴抽出
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)

        # 2. FCによる段階的圧縮
        h = self.encoder_fc(h)

        # 3. 潜在変数の計算と安全装置（クランプ）
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)

        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu

        # 4. FCによる段階的復元
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) # 4階テンソルに戻す

        # 5. CNNによる画像復元
        recon = self.decoder_conv(h_recon)

        return recon, mu, logvar



In [ ]:
def binary_roc_curve(y_true, scores):
    labels = np.asarray(y_true)
    scores = np.asarray(scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or labels.shape != scores.shape:
        raise ValueError('ラベルとスコアは同じ長さの1次元配列にしてください。')
    if not np.isfinite(scores).all() or not np.isin(labels, [0, 1]).all():
        raise ValueError('スコアは有限値、ラベルは0または1である必要があります。')
    positives = int((labels == 1).sum())
    negatives = int((labels == 0).sum())
    if not positives or not negatives:
        raise ValueError('正常と異常の両方が必要です。')
    order = np.argsort(-scores, kind='stable')
    ordered_scores = scores[order]
    ordered_labels = labels[order]
    ends = np.r_[np.flatnonzero(np.diff(ordered_scores)), len(scores) - 1]
    true_positives = np.cumsum(ordered_labels)[ends]
    false_positives = (ends + 1) - true_positives
    fpr = np.r_[0.0, false_positives / negatives]
    tpr = np.r_[0.0, true_positives / positives]
    thresholds = np.r_[np.inf, ordered_scores[ends]]
    return fpr, tpr, thresholds

def binary_auroc(y_true, scores):
    fpr, tpr, _ = binary_roc_curve(y_true, scores)
    return float(np.trapezoid(tpr, fpr))

assert binary_auroc([0,0,1,1], [0.1,0.2,0.8,0.9]) == 1.0
assert binary_auroc([0,0,1,1], [0.9,0.8,0.2,0.1]) == 0.0
assert binary_auroc([0,0,1,1], [0.5,0.5,0.5,0.5]) == 0.5



def verify_auroc(y,scores):
    value=binary_auroc(y,scores)
    normal=scores[y==0]; abnormal=scores[y==1]
    rank=((abnormal[:,None]>normal).sum()+0.5*(abnormal[:,None]==normal).sum())/(len(normal)*len(abnormal))
    assert np.isclose(value,rank,atol=1e-12)
    return value


In [ ]:
def score_loaded_model(model,images):
    model.eval()
    reconstructed = np.empty((len(images),128,128),dtype=np.float32)
    error_maps = np.empty_like(reconstructed)
    scores = np.empty(len(images),dtype=np.float64)
    with torch.inference_mode():
        for start in range(0,len(images),EVAL_BATCH_SIZE):
            x = images[start:start+EVAL_BATCH_SIZE].to(DEVICE)
            output,_,_ = model(x,sample=False)
            assert torch.isfinite(output).all()
            for local in range(len(x)):
                index = start+local
                a = x[local,0].cpu().numpy()
                b = output[local,0].cpu().numpy()
                _,ssim_map = ssim(a,b,data_range=1.0,win_size=11,full=True)
                error = 1.0-ssim_map
                assert np.isfinite(error).all()
                reconstructed[index] = b
                error_maps[index] = error
                scores[index] = np.sort(error.ravel())[::-1][:TOP_K].mean()
    return scores,reconstructed,error_maps


## 4. 全100画像×10角度を両モデルで評価

回転は元の1024×1024で行い、最後に128×128へ縮小します。0度では元画像と完全一致することを確認します。回転・前処理は角度ごとに1回だけ行い、その同じ入力を両モデルへ与えます。学習・逆伝播・重みの保存は行いません。

可視化には評価時の復元画像・誤差マップを保存し、図のための再推論を避けます。メモリと保存容量を抑えるため、画像配列は表示対象だけを保持します。全画像のスコアと入力画像の識別情報は保存します。


In [ ]:
run_id = datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
RESULT_DIR = OUTPUT_BASE/run_id
RESULT_DIR.mkdir(parents=True,exist_ok=False)
settings = {'experiment':'transistor_rotated_test_20261106_v1','run_id':run_id,'status':'running',
    'model_hashes':EXPECTED_HASHES,'source_hashes':source_hashes,'models':{k:p.relative_to(ROOT).as_posix() for k,p in MODEL_PATHS.items()},
    'angles':list(ANGLES),'assignment_seed':ASSIGNMENT_SEED,'test_manifest_sha256':manifest_hash,
    'normal_training_config':normal_config,'rotation_training_config':rotation_config,
    'preprocessing':'Full RGB1024 -> reflect pad -> NEAREST rotation -> central1024 -> preserve original-supported pixels -> LANCZOS128 -> grayscale -> ToTensor',
    'rotation_padding_width':rotation_pad_width((1024,1024)),
    'inference':'posterior mean, eval mode, no sampling','ssim_window':11,'data_range':1,'top_k':TOP_K,
    'device':str(DEVICE),'torch_threads':torch.get_num_threads(),
    'visual_paths':image_table.loc[visual_indices,'relative_path'].tolist(),'visual_angles':list(visual_angles),
    'label_assumption':'Global rotation models camera orientation; original labels retained.',
    'python':platform.python_version(),
    'versions':{v:importlib.metadata.version(v) for v in ['torch','torchvision','numpy','scikit-image','Pillow']}}
write_json(RESULT_DIR/'config.json',settings)
write_json(RESULT_DIR/'test_manifest.json',image_records)
assignment.to_csv(RESULT_DIR/'angle_assignment.csv',index=False)
models = {}
score_rows,rotated_records = [],[]
visual_cache = {}
try:
    for name,path in MODEL_PATHS.items():
        model = VAE_Hybrid_CNN_FC(latent_dim=64).to(DEVICE)
        model.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True),strict=True)
        model.eval()
        models[name] = model
    for angle in ANGLES:
        tensors = []
        for index,image in enumerate(base_images):
            transformed = to_tensor(final_transform(rotate_with_reflect(image,angle,verify_support=(index==0))))
            if angle == 0:
                assert torch.equal(transformed,base_tensor[index]), '0度の入力が以前と異なります。'
            tensors.append(transformed)
            rotated_records.append({'relative_path':image_table.loc[index,'relative_path'],
                'angle':angle,'source_sha256':image_records[index]['file_sha256'],
                'preprocessed_sha256':hashlib.sha256(transformed.numpy().tobytes()).hexdigest()})
        tensor = torch.stack(tensors)
        for name,model in models.items():
            scores,reconstructed,maps = score_loaded_model(model,tensor)
            if angle == 0:
                recorded = baseline_scores.loc[image_table.relative_path,name].to_numpy()
                assert np.allclose(scores,recorded,rtol=1e-5,atol=1e-6), '0度のスコアが保存済み結果と異なります。'
                assert np.isclose(verify_auroc(image_table.true_label.to_numpy(),scores),
                                  baseline_comparison.loc[name,'auroc'],rtol=0,atol=1e-12)
            for index,row in image_table.iterrows():
                score_rows.append({'condition':name,'angle':angle,'relative_path':row.relative_path,
                    'category':row.category,'category_ja':row.category_ja,'true_label':int(row.true_label),
                    'score':float(scores[index]),'mixed_selected':bool(assigned_angles[index]==angle)})
            if angle in visual_angles:
                for index in visual_indices:
                    key = (index,angle)
                    visual_cache.setdefault(key,{'input':tensor[index,0].numpy().copy()})
                    visual_cache[key][name] = {'reconstruction':reconstructed[index].copy(),'error_map':maps[index].copy(),
                                               'score':float(scores[index])}
            del reconstructed,maps
        pd.DataFrame(score_rows).to_csv(RESULT_DIR/'image_scores.csv',index=False)
        print(f'{angle}度：両モデルの評価完了',flush=True)
    scores_table = pd.DataFrame(score_rows)
    assert len(scores_table) == 2000
    assert not scores_table.duplicated(['condition','relative_path','angle']).any()
    write_json(RESULT_DIR/'rotated_input_manifest.json',rotated_records)
    settings['status'] = 'inference_completed'
    write_json(RESULT_DIR/'config.json',settings)
except BaseException as exc:
    settings.update(status='interrupted_or_failed',error=f'{type(exc).__name__}: {exc}')
    write_json(RESULT_DIR/'config.json',settings)
    raise
finally:
    models.clear()


## 5. 角度ごとのAUROC

横軸はテスト画像の回転角度、縦軸はAUROCです。各点は正常60枚・異常40枚から計算します。これは同じ元画像群の繰り返し評価で、独立した実験10回ではありません。


In [ ]:
angle_rows = []
for (name,angle),part in scores_table.groupby(['condition','angle'],sort=False):
    assert len(part) == 100
    angle_rows.append({'condition':name,'angle':int(angle),'auroc':verify_auroc(part.true_label.to_numpy(),part.score.to_numpy())})
angle_auroc = pd.DataFrame(angle_rows)
display(angle_auroc.pivot(index='angle',columns='condition',values='auroc').rename(columns=LABELS))
fig,ax = plt.subplots(figsize=(11,6),constrained_layout=True)
for name in CONDITIONS:
    part = angle_auroc[angle_auroc.condition==name].sort_values('angle')
    ax.plot(part.angle,part.auroc,marker='o',color=COLORS[name],label=LABELS[name])
ax.set(xlabel='検証画像の回転角度（度）',ylabel='AUROC',ylim=(0,1.05),xticks=ANGLES,
       title='画像の向きが変わったとき、異常検知性能はどう変わるか')
ax.grid(ls='--',alpha=.5);ax.legend()
fig.savefig(RESULT_DIR/'angle_auroc.png',dpi=180);plt.show();plt.close(fig)
angle_auroc.to_csv(RESULT_DIR/'angle_auroc.csv',index=False)


## 6. 回転なし／向きが混在／全角度（補足）の比較

「向きが混在」は1画像1角度の100枚です。「全角度（補足）」は延べ1,000枚をまとめてスコアの順位を評価します。後者には異なる角度間での正常・異常の比較も入るため、角度別AUROCの単純平均とは一致するとは限りません。


In [ ]:
scenario_parts = {}
scenario_rows = []
for name in CONDITIONS:
    full = scores_table[scores_table.condition==name]
    for scenario,part in [('unrotated',full[full.angle==0]),('mixed',full[full.mixed_selected]),('pooled',full)]:
        assert len(part) == (1000 if scenario=='pooled' else 100)
        if scenario != 'pooled':
            assert part.relative_path.nunique() == 100
        scenario_parts[(name,scenario)] = part
        scenario_rows.append({'condition':name,'scenario':scenario,'sample_count':len(part),'source_count':part.relative_path.nunique(),
                              'auroc':verify_auroc(part.true_label.to_numpy(),part.score.to_numpy())})
scenario_comparison = pd.DataFrame(scenario_rows)
display(scenario_comparison.replace({'condition':LABELS,'scenario':SCENARIO_LABELS}))
fig,ax = plt.subplots(figsize=(11,6),constrained_layout=True)
x = np.arange(len(SCENARIO_LABELS));width = .35
for offset,name in enumerate(CONDITIONS):
    values = scenario_comparison[scenario_comparison.condition==name].set_index('scenario').loc[list(SCENARIO_LABELS),'auroc'].to_numpy()
    positions = x+(offset-.5)*width
    ax.bar(positions,values,width=width,label=LABELS[name],color=COLORS[name])
    for px,value in zip(positions,values):
        ax.text(px,value+.015,f'{value:.4f}',ha='center',fontsize=11)
ax.set(xticks=x,xticklabels=list(SCENARIO_LABELS.values()),ylabel='AUROC',ylim=(0,1.1),title='テスト画像の向きによるAUROCの比較')
ax.grid(axis='y',ls='--',alpha=.5);ax.set_axisbelow(True);ax.legend()
fig.savefig(RESULT_DIR/'scenario_auroc.png',dpi=180);plt.show();plt.close(fig)
scenario_comparison.to_csv(RESULT_DIR/'scenario_comparison.csv',index=False)


## 7. 各条件のROC曲線とスコア分布（閾値探索なし）


In [ ]:
for scenario,title in SCENARIO_LABELS.items():
    fig,ax = plt.subplots(figsize=(8,8),constrained_layout=True)
    for name in CONDITIONS:
        part = scenario_parts[(name,scenario)]
        fpr,tpr,_ = binary_roc_curve(part.true_label.to_numpy(),part.score.to_numpy())
        auc = verify_auroc(part.true_label.to_numpy(),part.score.to_numpy())
        ax.plot(fpr,tpr,color=COLORS[name],label=f'{LABELS[name]} (AUROC={auc:.4f})',lw=2)
    ax.set(xlim=(0,1),ylim=(0,1.05),xlabel='False Positive Rate（正常を誤検出した割合）',
           ylabel='True Positive Rate（異常を検出した割合）',title=f'ROC：{title}')
    ax.legend(loc='lower right');ax.grid(ls='--',alpha=.6)
    fig.savefig(RESULT_DIR/f'roc_{scenario}.png',dpi=180);plt.show();plt.close(fig)
    joined = np.concatenate([scenario_parts[(name,scenario)].score.to_numpy() for name in CONDITIONS])
    lo,hi = float(joined.min()),float(joined.max())
    if lo == hi:
        lo-=1e-6;hi+=1e-6
    bins = np.linspace(lo,hi,40)
    ymax = max(np.histogram(scenario_parts[(name,scenario)].loc[scenario_parts[(name,scenario)].true_label==label,'score'],bins=bins)[0].max()
               for name in CONDITIONS for label in [0,1])+1
    for name in CONDITIONS:
        part = scenario_parts[(name,scenario)]
        fig,ax = plt.subplots(figsize=(10,6),constrained_layout=True)
        for label,color,text in [(0,'green','正常'),(1,'red','異常')]:
            ax.hist(part.loc[part.true_label==label,'score'],bins=bins,color=color,alpha=.5,edgecolor='white',label=text)
        ax.set(xlabel='異常スコア（SSIM誤差上位130点の平均）',ylabel='評価サンプル数',
               xlim=(lo,hi),ylim=(0,ymax),title=f'{title} / {LABELS[name]}')
        ax.legend();ax.grid(axis='y',ls='--',alpha=.5)
        fig.savefig(RESULT_DIR/f'hist_{scenario}_{name}.png',dpi=180);plt.show();plt.close(fig)


## 8. 異常種類別の比較

各角度では正常60枚と各異常10枚の比較です。配置ずれも元ラベルのままであり、画像全体の回転と異常の意味の関係に注意して解釈します。


In [ ]:
category_rows = []
category_scenarios = []
for category in CATEGORY_LABELS:
    if category == 'good':
        continue
    for name in CONDITIONS:
        for angle in ANGLES:
            part = scores_table[(scores_table.condition==name)&(scores_table.angle==angle)&scores_table.category.isin(['good',category])]
            category_rows.append({'condition':name,'category':category,'category_ja':CATEGORY_LABELS[category],'angle':angle,
                'normal_count':60,'anomaly_count':10,'auroc':verify_auroc(part.true_label.to_numpy(),part.score.to_numpy())})
        for scenario in SCENARIO_LABELS:
            full = scenario_parts[(name,scenario)]
            part = full[full.category.isin(['good',category])]
            category_scenarios.append({'condition':name,'category':category,'category_ja':CATEGORY_LABELS[category],
                'scenario':scenario,'sample_count':len(part),'source_count':part.relative_path.nunique(),
                'auroc':verify_auroc(part.true_label.to_numpy(),part.score.to_numpy())})
category_auroc = pd.DataFrame(category_rows)
category_scenario_auroc = pd.DataFrame(category_scenarios)
display(category_scenario_auroc.replace({'condition':LABELS,'scenario':SCENARIO_LABELS}))
for name in CONDITIONS:
    matrix = category_auroc[category_auroc.condition==name].pivot(index='category_ja',columns='angle',values='auroc').loc[:,list(ANGLES)]
    fig,ax = plt.subplots(figsize=(12,5),constrained_layout=True)
    heat = ax.imshow(matrix.to_numpy(),vmin=0,vmax=1,cmap='viridis',aspect='auto')
    ax.set_xticks(np.arange(len(ANGLES)),ANGLES);ax.set_yticks(np.arange(len(matrix)),matrix.index)
    ax.set(xlabel='検証画像の回転角度（度）',title=f'異常種類別AUROC / {LABELS[name]}')
    for row in range(len(matrix)):
        for col in range(len(ANGLES)):
            value = matrix.iloc[row,col]
            ax.text(col,row,f'{value:.3f}',ha='center',va='center',color='black' if value>.65 else 'white',fontsize=10)
    fig.colorbar(heat,ax=ax,label='AUROC')
    fig.savefig(RESULT_DIR/f'category_angles_{name}.png',dpi=180);plt.show();plt.close(fig)
category_auroc.to_csv(RESULT_DIR/'category_angle_auroc.csv',index=False)
category_scenario_auroc.to_csv(RESULT_DIR/'category_scenario_auroc.csv',index=False)


## 9. 回転前・回転後の入力／復元／SSIM誤差マップ

評価に使用した復元と誤差マップをそのまま表示します。明るい部分は入力と復元の違いで、欠陥の正解位置ではありません。誤差の色範囲は全図共通0〜2です。


In [ ]:
visual_dir = RESULT_DIR/'visualizations'
visual_dir.mkdir(exist_ok=True)
for (index,angle),cache in visual_cache.items():
    record = image_table.loc[index]
    fig,axes = plt.subplots(2,4,figsize=(13,7),constrained_layout=True)
    for row,name in enumerate(CONDITIONS):
        axes[row,0].imshow(base_tensor[index,0],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
        axes[row,1].imshow(cache['input'],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
        axes[row,2].imshow(cache[name]['reconstruction'],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
        heat = axes[row,3].imshow(cache[name]['error_map'],cmap='inferno',vmin=0,vmax=2,interpolation='nearest')
        axes[row,0].set_ylabel(LABELS[name])
        for col,title in enumerate(['回転前の入力',f'{angle}度回転した入力',f'復元 / スコア={cache[name]["score"]:.4f}','SSIM誤差マップ']):
            axes[row,col].set_title(title,fontsize=11)
            axes[row,col].set_xticks([]);axes[row,col].set_yticks([])
    fig.colorbar(heat,ax=axes[:,3],label='1−SSIM（色範囲は全図共通）',shrink=.8)
    fig.suptitle(f'{record.relative_path} / 元の正解：{record.category_ja} / 平均μから復元',fontsize=14)
    stem = record.relative_path.replace('/','_').replace('.png','')+f'_{angle:03d}deg'
    fig.savefig(visual_dir/f'{stem}.png',dpi=160)
    plt.show();plt.close(fig)
    # 表示に使った配列を保存し、後から再推論せず取り出せるようにします。
    np.savez_compressed(visual_dir/f'{stem}.npz',original=base_tensor[index,0].numpy(),rotated=cache['input'],
        **{f'{name}_{key}':cache[name][key] for name in CONDITIONS for key in ['reconstruction','error_map']})
print('表示画像の設定を変える場合は、設定セルを変更して上から実行してください。別の評価フォルダへ保存されます。')


## 10. 保存と元ファイルの不変確認

結果は新規フォルダへ保存します。元画像と保存済みモデル・設定・比較結果は上書きしません。グラフの改善・低下の原因は、向き、背景や端の加工、欠陥の見え方などが混ざるため、結果だけから1つに断定しません。

回転拡張の学習時に含まれた10方向と同じ角度での評価です。未知の角度や実撮影での回転に対する一般化までを保証しません。


In [ ]:
assert all(sha256(ROOT/path) == value for path,value in source_hashes.items()), '読み込み元が実行中に変更されました。'
assert all(sha256(ROOT/r['repo_path']) == r['file_sha256'] for r in image_records), '元画像が実行中に変更されました。'
settings.update(status='completed',completed_datetime_jst=datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
    unique_test_images=100,evaluation_variants=1000,model_score_count=len(scores_table),
    baseline_auroc_verified=True,scenario_results=scenario_comparison.to_dict('records'),
    limitations=['synthetic rotation includes padding/cropping artifacts','original anomaly labels retained by assumption',
                 'same 100 source images reused','single training seed and single balanced angle assignment',
                 'angles seen in rotation training','exploratory evaluation after viewing baseline results'])
write_json(RESULT_DIR/'config.json',settings)
print('結果保存先：',RESULT_DIR)
print('回転なし・角度別・向きが混在する場合のAUROCを区別して確認してください。')


## 参照・確認状態

- [更新回数を揃えた比較](261106transistorの更新回数を揃えた通常学習と回転拡張のAUROC比較.ipynb)：モデル構造・平均推論・SSIM Top-130・既存の0度スコア。
- 回転処理は保存済み学習用ノートブックの関数を継承。処理順・補完方法は保存設定に明記します。報告用の図表名称は「通常学習」「回転拡張」です。
- コード作成と静的確認のみ。アシスタントは学習・推論・数値テスト・描画を実行していません。評価結果と図の表示はユーザー実行後に確認してください。
